# 02 - Exploratory Data Analysis

**Kenya Food Price Inflation Tracker**

## Objective
Explore and visualize food price trends, patterns, and relationships in the cleaned dataset.

## Key Questions
1. What are the long-term price trends for staple commodities?
2. Which commodities show highest volatility?
3. Are there seasonal patterns?
4. How do prices vary by region?
5. What is the inflation rate over time?

In [4]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 1. Load Cleaned Data

In [5]:
# Load cleaned datasets
try:
    df_staples = pd.read_csv('../data/clean/wfp_core_staples.csv', parse_dates=['date'])
except FileNotFoundError:
    df_staples = pd.read_csv('data/clean/wfp_core_staples.csv', parse_dates=['date'])

try:
    df_monthly = pd.read_csv('../data/clean/wfp_monthly_avg.csv', parse_dates=['date'])
except FileNotFoundError:
    df_monthly = pd.read_csv('data/clean/wfp_monthly_avg.csv', parse_dates=['date'])

print(f"Staples dataset: {df_staples.shape}")
print(f"Monthly dataset: {df_monthly.shape}")
print(f"\nDate range: {df_staples['date'].min()} to {df_staples['date'].max()}")
print(f"Commodities: {df_staples['cm_name'].nunique()}")
print(f"Markets: {df_staples['mkt_name'].nunique()}")


Staples dataset: (4160, 19)
Monthly dataset: (4160, 9)

Date range: 2006-01-01 00:00:00 to 2021-08-01 00:00:00
Commodities: 9
Markets: 45


## 2. Price Trends Over Time

In [6]:
# Aggregate prices by commodity and date (national average)
national_avg = df_staples.groupby(['date', 'cm_name'])['mp_price'].mean().reset_index()
national_avg.columns = ['date', 'commodity', 'price']

# Interactive time series plot
fig = px.line(national_avg, x='date', y='price', color='commodity',
              title='Kenya Food Price Trends (2006-2021)',
              labels={'price': 'Price (KES)', 'date': 'Date'},
              height=600)
fig.update_layout(hovermode='x unified')
fig.show()

print("✓ Time series plot created")


✓ Time series plot created


In [7]:
# Focus on Maize (most important staple)
maize_data = national_avg[national_avg['commodity'] == 'Maize (white) - Retail'].copy()

fig = px.line(maize_data, x='date', y='price',
              title='Maize (White) Retail Price - National Average',
              labels={'price': 'Price (KES/kg)', 'date': 'Date'})
fig.add_hline(y=maize_data['price'].mean(), line_dash='dash',
              annotation_text=f'Mean: {maize_data["price"].mean():.2f} KES')
fig.show()

print(f"Maize price statistics:")
print(f"Mean: {maize_data['price'].mean():.2f} KES")
print(f"Median: {maize_data['price'].median():.2f} KES")
print(f"Std Dev: {maize_data['price'].std():.2f} KES")
print(f"Min: {maize_data['price'].min():.2f} KES ({maize_data[maize_data['price'] == maize_data['price'].min()]['date'].values[0]})")
print(f"Max: {maize_data['price'].max():.2f} KES ({maize_data[maize_data['price'] == maize_data['price'].max()]['date'].values[0]})")

Maize price statistics:
Mean: 46.21 KES
Median: 49.96 KES
Std Dev: 13.08 KES
Min: 18.50 KES (2007-03-01T00:00:00.000000000)
Max: 68.54 KES (2017-06-01T00:00:00.000000000)


## 3. Price Volatility Analysis

In [8]:
# Calculate coefficient of variation (CV) for each commodity
volatility = national_avg.groupby('commodity')['price'].agg(['mean', 'std'])
volatility['cv'] = (volatility['std'] / volatility['mean']) * 100
volatility = volatility.sort_values('cv', ascending=False)

print("Commodity Price Volatility (Coefficient of Variation):")
print(volatility)

# Plot volatility
fig = px.bar(volatility.reset_index(), x='commodity', y='cv',
             title='Price Volatility by Commodity (CV %)',
             labels={'cv': 'Coefficient of Variation (%)', 'commodity': 'Commodity'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

Commodity Price Volatility (Coefficient of Variation):
                                        mean        std         cv
commodity                                                         
Maize - Wholesale                  27.906097   8.393574  30.077921
Beans (dry) - Retail               77.536191  22.084897  28.483340
Maize (white) - Retail             46.207877  13.076898  28.300149
Milk (cow, pasteurized) - Retail   37.175787  10.336617  27.804703
Beans - Wholesale                  60.258151  12.572577  20.864525
Oil (vegetable) - Retail          181.811721  34.843039  19.164352
Maize - Retail                     47.233541   2.512207   5.318692
Rice - Retail                      98.367905   3.108235   3.159806
Sugar - Retail                    109.193258   2.431229   2.226538


## 4. Seasonal Patterns

In [9]:
# Extract month from date
# Ensure date is datetime for .dt operations
national_avg['date'] = pd.to_datetime(national_avg['date'])

national_avg['month'] = national_avg['date'].dt.month
national_avg['year'] = national_avg['date'].dt.year

# Monthly average prices (across all years)
seasonal = national_avg.groupby(['commodity', 'month'])['price'].mean().reset_index()

# Plot seasonal patterns for key commodities
key_commodities = ['Maize (white) - Retail', 'Beans (dry) - Retail', 'Rice - Retail']
seasonal_key = seasonal[seasonal['commodity'].isin(key_commodities)]

fig = px.line(seasonal_key, x='month', y='price', color='commodity',
              title='Seasonal Price Patterns (Average across years)',
              labels={'price': 'Average Price (KES)', 'month': 'Month'},
              markers=True)
fig.update_xaxes(tickmode='linear', tick0=1, dtick=1)
fig.show()


## 5. Regional Price Comparison

In [10]:
# Regional average prices for Maize
maize_regional = df_staples[
    df_staples['cm_name'] == 'Maize (white) - Retail'
].groupby('adm1_name')['mp_price'].agg(['mean', 'std', 'count']).reset_index()
maize_regional.columns = ['region', 'avg_price', 'std_price', 'observations']
maize_regional = maize_regional.sort_values('avg_price', ascending=False)

print("Maize Prices by Region:")
print(maize_regional)

fig = px.bar(maize_regional, x='region', y='avg_price',
             title='Average Maize Prices by Region',
             labels={'avg_price': 'Average Price (KES/kg)', 'region': 'Region'},
             error_y='std_price')
fig.update_layout(xaxis_tickangle=-45)
fig.show()


Maize Prices by Region:
          region  avg_price  std_price  observations
2  North Eastern  58.155101  16.546705           247
3    Rift Valley  55.633048  16.506164           315
0          Coast  47.180288   9.842891           139
1        Eastern  35.926638  12.228165           351


In [11]:
# Regional time series for Maize
maize_regional_ts = df_staples[
    df_staples['cm_name'] == 'Maize (white) - Retail'
].groupby(['date', 'adm1_name'])['mp_price'].mean().reset_index()
maize_regional_ts.columns = ['date', 'region', 'price']

fig = px.line(maize_regional_ts, x='date', y='price', color='region',
              title='Maize Price Trends by Region',
              labels={'price': 'Price (KES/kg)', 'date': 'Date'})
fig.show()


## 6. Inflation Rate Calculation

In [12]:
# Calculate Year-over-Year (YoY) inflation for Maize
maize_yoy = maize_data.copy()
maize_yoy = maize_yoy.sort_values('date')
maize_yoy['price_lag_12m'] = maize_yoy['price'].shift(12)
maize_yoy['inflation_yoy'] = ((maize_yoy['price'] - maize_yoy['price_lag_12m']) / maize_yoy['price_lag_12m']) * 100

# Drop first 12 months (no comparison data)
maize_yoy = maize_yoy.dropna()

# Plot YoY inflation
fig = go.Figure()
fig.add_trace(go.Scatter(x=maize_yoy['date'], y=maize_yoy['inflation_yoy'],
                         mode='lines', name='YoY Inflation',
                         line=dict(color='royalblue')))
fig.add_hline(y=0, line_dash='dash', line_color='red')
fig.update_layout(title='Maize Price - Year-over-Year Inflation Rate',
                  xaxis_title='Date',
                  yaxis_title='Inflation Rate (%)',
                  hovermode='x unified')
fig.show()

print(f"\nMaize Inflation Statistics:")
print(f"Average YoY inflation: {maize_yoy['inflation_yoy'].mean():.2f}%")
print(f"Median YoY inflation: {maize_yoy['inflation_yoy'].median():.2f}%")
print(f"Max YoY inflation: {maize_yoy['inflation_yoy'].max():.2f}% ({maize_yoy[maize_yoy['inflation_yoy'] == maize_yoy['inflation_yoy'].max()]['date'].values[0]})")
print(f"Min YoY inflation: {maize_yoy['inflation_yoy'].min():.2f}% ({maize_yoy[maize_yoy['inflation_yoy'] == maize_yoy['inflation_yoy'].min()]['date'].values[0]})")


Maize Inflation Statistics:
Average YoY inflation: 9.22%
Median YoY inflation: 0.95%
Max YoY inflation: 117.47% (2017-04-01T00:00:00.000000000)
Min YoY inflation: -41.14% (2016-04-01T00:00:00.000000000)


## 7. Summary Statistics Table

In [13]:
# Comprehensive summary statistics by commodity
summary = national_avg.groupby('commodity')['price'].agg([
    'count', 'mean', 'median', 'std', 'min', 'max'
]).round(2)
summary['cv'] = ((summary['std'] / summary['mean']) * 100).round(2)
summary = summary.sort_values('mean', ascending=False)

print("\n=== COMPREHENSIVE SUMMARY STATISTICS ===")
print(summary)

# Save summary
import os
os.makedirs('../reports', exist_ok=True)
summary.to_csv('../reports/eda_summary_statistics.csv')
print("\n✓ Summary statistics saved to reports/eda_summary_statistics.csv")



=== COMPREHENSIVE SUMMARY STATISTICS ===
                                  count    mean  median    std     min  \
commodity                                                                
Oil (vegetable) - Retail            186  181.81  185.00  34.84  108.00   
Sugar - Retail                        9  109.19  108.76   2.43  106.22   
Rice - Retail                         9   98.37   97.64   3.11   93.86   
Beans (dry) - Retail                188   77.54   80.00  22.08   23.00   
Beans - Wholesale                   168   60.26   60.87  12.57   30.17   
Maize - Retail                        9   47.23   45.87   2.51   44.40   
Maize (white) - Retail              180   46.21   49.96  13.08   18.50   
Milk (cow, pasteurized) - Retail    178   37.18   43.00  10.34   20.00   
Maize - Wholesale                   186   27.91   29.45   8.39   12.62   

                                     max     cv  
commodity                                        
Oil (vegetable) - Retail          250.00  1

## Key Findings

### 1. Long-term Trends
- Maize prices show an upward trend from ~35 KES/kg (2006) to ~50 KES/kg (2021)
- Sharp price spikes in 2011 and 2017 (likely due to drought)

### 2. Volatility
- Maize shows highest absolute volatility
- Imported goods (rice, oil) show relatively stable prices

### 3. Seasonality
- Clear seasonal patterns in maize prices
- Harvest seasons (Oct-Dec) show lower prices
- Lean seasons (Feb-May) show price peaks

### 4. Regional Variation
- Nairobi and Coast regions show higher average prices
- Rift Valley (production region) shows lower prices
- Price gap of 20-30 KES between regions

### 5. Inflation
- Average YoY inflation: ~9%
- Highest inflation spike: 117% (April 2017)
- Periods of deflation during bumper harvests